# A Transformer Without Training: 40 Digits, Exactly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/transformer_by_hand.ipynb)

Companion notebook to [the post](https://sesen.ai/blog/transformer-without-training).

Nothing here is trained. Every weight is written down by a function that knows
what computation it wants, and the forward pass is the ordinary transformer
forward pass. Three constructions of rising ambition:

1. **Reverse a sequence**, one attention head, no MLP
2. **Count a token**, one head plus one MLP, and the length trap that comes with it
3. **Add multi-digit numbers**, two blocks, one live head, exact at any length

Then we train a transformer of the same shape on the same task and find the cliff.

Runtime: about three minutes, most of it the training run at the end.

## 1. The forward pass

Standard attention, standard MLP, standard residual stream. The only unusual choice is the mask, which comes in a strict flavour used by the adder.

In [ ]:
import numpy as np

def attend(x, w_q, w_k, w_v, w_o, mask="causal"):
    n = x.shape[0]
    q, k, v = x @ w_q, x @ w_k, x @ w_v
    scores = q @ k.T
    if mask != "none":
        i, j = np.indices((n, n))
        allowed = j <= i if mask == "causal" else j < i
        if mask == "strict":
            allowed[0, 0] = True          # row 0 has no earlier key; let it hold still
        scores = np.where(allowed, scores, -np.inf)
    scores = scores - scores.max(axis=-1, keepdims=True)
    w = np.exp(scores)
    w /= w.sum(axis=-1, keepdims=True)
    return w @ v @ w_o, w

## 2. Reversing a sequence

The query at position `i` is the one-hot for position `n-1-i`; the key at position
`j` is the one-hot for `j`. Their dot product is 1 at the mirror position and 0
everywhere else, and `beta` turns that into a hard selection.

In [ ]:
def build_reverser(vocab=10, n=8, beta=100.0):
    d = vocab + n + vocab                        # token block, position block, output block
    w_e = np.zeros((vocab, d)); w_e[:, :vocab] = np.eye(vocab)
    w_p = np.zeros((n, d));     w_p[:, vocab:vocab + n] = np.eye(n)

    w_q = np.zeros((d, n))
    for i in range(n):
        w_q[vocab + i, n - 1 - i] = beta         # position i asks for position n-1-i
    w_k = np.zeros((d, n));     w_k[vocab:vocab + n] = np.eye(n)
    w_v = np.zeros((d, vocab)); w_v[:vocab] = np.eye(vocab)
    w_o = np.zeros((vocab, d)); w_o[:, -vocab:] = np.eye(vocab)
    w_u = np.zeros((d, vocab)); w_u[-vocab:] = np.eye(vocab)
    return w_e, w_p, (w_q, w_k, w_v, w_o), w_u

w_e, w_p, head, w_u = build_reverser()
seq = [3, 1, 4, 1, 5, 9, 2, 6]

x = w_e[seq] + w_p                               # the residual stream
delta, pattern = attend(x, *head, mask="none")
print(((x + delta) @ w_u).argmax(-1))

Every row of the pattern puts weight 1.00 on exactly one column.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.imshow(pattern, cmap="GnBu", vmin=0, vmax=1)
ax.set_xticks(range(8), seq); ax.set_yticks(range(8), seq[::-1])
ax.set_xlabel("attends to position j"); ax.set_ylabel("output at position i")
ax.set_title("a permutation matrix, written by hand")
plt.show()

### The temperature knob

Softmax is a soft argmax. Scaling `w_q` by `beta` scales every score, which is the
same as lowering the temperature. How high `beta` has to go depends on what reads
the result: the reverser takes an argmax and tolerates a blurred pattern, while
the adder feeds its attention output into arithmetic and does not.

In [ ]:
for beta in (0.25, 0.5, 0.75, 2.0, 8.0):
    w_e, w_p, head, w_u = build_reverser(beta=beta)
    x = w_e[seq] + w_p
    delta, pat = attend(x, *head, mask="none")
    out = ((x + delta) @ w_u).argmax(-1).tolist()
    mass = pat[np.arange(8), 7 - np.arange(8)].mean()
    print(f"beta {beta:5.2f}  mass on target {mass:.3f}  exact {out == seq[::-1]}")

## 3. Counting, and the length trap

Set the query and key matrices to zero. Every score is 0, softmax spreads
attention evenly, and the head returns the plain **mean** of its values. A mean is
not a sum, so recovering a count needs the sequence length, and any length you
bake in is a length you will later fail at.

In [ ]:
def build_counter(target, vocab=10, max_len=16, seq_len=8):
    d = vocab + 1 + (vocab + 1)
    frac, count = vocab, vocab + 1
    w_e = np.zeros((vocab, d)); w_e[:, :vocab] = np.eye(vocab)
    w_p = np.zeros((max_len, d))

    w_q = w_k = np.zeros((d, 1))                 # every score is zero: uniform attention
    w_v = np.zeros((d, 1)); w_v[target, 0] = 1.0
    w_o = np.zeros((1, d)); w_o[0, frac] = 1.0

    # frac * seq_len is the count; three ReLUs per value turn that scalar into a
    # one-hot bump that is 1 at k and 0 at every other integer.
    n_neurons = 3 * (vocab + 1)
    w_in, b_in = np.zeros((d, n_neurons)), np.zeros(n_neurons)
    w_out = np.zeros((n_neurons, d))
    for k in range(vocab + 1):
        for t, offset in enumerate((-1, 0, 1)):
            w_in[frac, 3 * k + t] = seq_len
            b_in[3 * k + t] = -(k + offset)
        w_out[3 * k + 0, count + k] = 1.0
        w_out[3 * k + 1, count + k] = -2.0
        w_out[3 * k + 2, count + k] = 1.0

    w_u = np.zeros((d, vocab + 1)); w_u[count:, :] = np.eye(vocab + 1)
    return (w_e, w_p, (w_q, w_k, w_v, w_o), (w_in, b_in, w_out), w_u), frac

parts, frac = build_counter(target=7, seq_len=8)
w_e, w_p, head, mlp, w_u = parts

def count_tokens(seq):
    x = w_e[seq] + w_p[:len(seq)]
    x = x + attend(x, *head, mask="none")[0]
    fraction = x[0, frac]
    x = x + np.maximum(x @ mlp[0] + mlp[1], 0) @ mlp[2]
    return fraction, int((x @ w_u).argmax(-1)[0])

rng = np.random.default_rng(3)
for n, want in ((8, 1), (8, 3), (12, 2), (12, 3), (16, 3)):
    while True:                          # pick a sequence with a readable count
        seq = rng.integers(0, 10, n).tolist()
        if sum(s == 7 for s in seq) == want:
            break
    fraction, predicted = count_tokens(seq)
    print(f"n={n:2d}  true {want}  head writes {fraction:.4f}  reads out {predicted}")

At `n = 8`, the length the MLP was built for, the read-out is right. At any
other length it is wrong by exactly the ratio of the lengths. Hold onto that: the
same failure shows up at the end of this notebook with a cause we cannot name.

## 4. Addition, with carry lookahead

School addition ripples: add a column, carry, repeat. A ripple of `n` steps needs
`n` sequential operations and a fixed-depth transformer does not have `n` layers.

Carry lookahead removes the recursion. Define two bits per column, where
`t = a + b` is the column sum before any carry:

- **generate**: `t >= 10`, this column carries whatever arrives
- **propagate**: `t == 9`, this column passes on whatever arrives

Then the carry into column `i` is the `generate` bit of the nearest column below
`i` that does not propagate. "Nearest earlier column satisfying a predicate" is
exactly one attention head.

In [ ]:
class Registers:
    def __init__(self):
        self.slots, self.width = {}, 0
    def alloc(self, name, size=1):
        self.slots[name] = slice(self.width, self.width + size)
        self.width += size
    def __getitem__(self, name):
        return self.slots[name].start

BOS = 100          # digit pairs (a, b) are tokens 10*a + b; 100 is the sentinel

def build_adder(max_len=64, beta=4000.0):
    reg = Registers()
    for name in ("one", "a", "b", "idx", "sum", "gen", "prop", "carry", "digit"):
        reg.alloc(name)
    d, eps = reg.width, 1.0 / (2 * max_len)

    w_e = np.zeros((101, d)); w_e[:, reg["one"]] = 1.0
    for a in range(10):
        for b in range(10):
            w_e[10 * a + b, reg["a"]] = a
            w_e[10 * a + b, reg["b"]] = b
    w_p = np.zeros((max_len, d)); w_p[:, reg["idx"]] = eps * np.arange(max_len)

    # MLP 0: sum, then two indicators built from ReLU ramps.
    #   gen  = relu(t-9) - relu(t-10)                      is 1 exactly when t >= 10
    #   prop = relu(t-8) - 2 relu(t-9) + relu(t-10)        is 1 exactly when t == 9
    w_in0, b_in0 = np.zeros((d, 4)), np.array([0.0, -8.0, -9.0, -10.0])
    w_in0[[reg["a"], reg["b"]], :] = 1.0
    w_out0 = np.zeros((4, d))
    w_out0[0, reg["sum"]] = 1.0                  # digits are non-negative, so relu(t) = t
    w_out0[[2, 3], reg["gen"]] = [1.0, -1.0]
    w_out0[[1, 2, 3], reg["prop"]] = [1.0, -2.0, 1.0]

    # Block 1 attention: every column asks the same question, so the query is
    # constant. The key ranks columns by (not propagating, then position).
    w_q = np.zeros((d, 2)); w_q[reg["one"], :] = beta
    w_k = np.zeros((d, 2))
    w_k[reg["one"], 0], w_k[reg["prop"], 0] = 1.0, -1.0
    w_k[reg["idx"], 1] = 1.0
    w_v = np.zeros((d, 1)); w_v[reg["gen"], 0] = 1.0
    w_o = np.zeros((1, d)); w_o[0, reg["carry"]] = 1.0

    # MLP 1: digit = u - 10 * 1[u >= 10], where u = sum + carry
    w_in1, b_in1 = np.zeros((d, 3)), np.array([0.0, -9.0, -10.0])
    w_in1[[reg["sum"], reg["carry"]], :] = 1.0
    w_out1 = np.zeros((3, d))
    w_out1[[0, 1, 2], reg["digit"]] = [1.0, -10.0, 10.0]

    # Unembed: logit_k = 2k*digit - k^2 argmaxes to the nearest integer, because
    # the -digit^2 term of -(digit - k)^2 is the same for every k.
    w_u = np.zeros((d, 10)); w_u[reg["digit"], :] = 2.0 * np.arange(10)
    b_u = -(np.arange(10.0) ** 2)
    return dict(reg=reg, w_e=w_e, w_p=w_p, head=(w_q, w_k, w_v, w_o),
                mlp0=(w_in0, b_in0, w_out0), mlp1=(w_in1, b_in1, w_out1), w_u=w_u, b_u=b_u)

The forward pass. Block 0 is MLP-only (its head would be a no-op, so we leave it out); block 1 is the carry lookup followed by the digit MLP.

In [ ]:
def run_adder(m, tokens, want_pattern=False):
    x = m["w_e"][tokens] + m["w_p"][:len(tokens)]
    x = x + np.maximum(x @ m["mlp0"][0] + m["mlp0"][1], 0) @ m["mlp0"][2]
    delta, pattern = attend(x, *m["head"], mask="strict")
    x = x + delta
    x = x + np.maximum(x @ m["mlp1"][0] + m["mlp1"][1], 0) @ m["mlp1"][2]
    digits = (x @ m["w_u"] + m["b_u"]).argmax(-1)
    return (digits, pattern, x) if want_pattern else digits

def pair_tokens(x, y, width):
    return [BOS] + [10 * ((x // 10**i) % 10) + ((y // 10**i) % 10) for i in range(width)]

def add(m, x, y, width):
    digits = run_adder(m, pair_tokens(x, y, width))[1:]
    return int("".join(str(d) for d in digits[::-1]))

adder = build_adder()
print(add(adder, 4999, 1, width=5))

A carry born in the lowest column has to travel three columns to reach its destination. Watch where the columns look.

In [ ]:
tokens = pair_tokens(4999, 1, 5)
digits, pattern, x = run_adder(adder, tokens, want_pattern=True)
labels = ["BOS"] + [f"{t // 10}+{t % 10}" for t in tokens[1:]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.imshow(pattern, cmap="GnBu", vmin=0, vmax=1)
ax1.set_xticks(range(6), labels); ax1.set_yticks(range(6), labels)
ax1.set_title("columns 2, 3 and 4 all read column 1")
ax1.set_xlabel("attends to column j")

names = list(adder["reg"].slots)
ax2.imshow(np.round(x[:, :len(names)].T, 3), cmap="GnBu", vmin=0, vmax=10)
ax2.set_yticks(range(len(names)), names); ax2.set_xticks(range(6), labels)
for (i, j), v in np.ndenumerate(np.round(x[:, :len(names)].T, 3)):
    ax2.text(j, i, f"{v:g}", ha="center", va="center", fontsize=8)
ax2.set_title("the residual stream at the end")
plt.tight_layout(); plt.show()

Nothing in the computation refers to the length of the input, so it runs at any width the positional table covers.

In [ ]:
rng = np.random.default_rng(11)
operand = lambda w: int("".join(str(d) for d in rng.integers(0, 10, w)))

for w in (4, 8, 20, 40):
    hits = 0
    for _ in range(200):
        p, q = operand(w), operand(w)
        hits += add(adder, p, q, w + 1) == p + q
    print(f"width {w:2d}: exact match {hits / 200:.3f}")

## 5. The same architecture, trained

A hand-built circuit proves the architecture **can** represent the algorithm. It
says nothing about whether gradient descent **finds** it. So train one: same
tokens, same task, two blocks, one head, random initialisation.

This cell takes about two minutes on a Colab CPU.

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(0)

class Learned(nn.Module):
    def __init__(self, d_model=64, max_len=64, n_blocks=2):
        super().__init__()
        self.tok, self.pos = nn.Embedding(101, d_model), nn.Embedding(max_len, d_model)
        self.attn = nn.ModuleList([nn.MultiheadAttention(d_model, 1, batch_first=True)
                                   for _ in range(n_blocks)])
        self.mlp = nn.ModuleList([nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.ReLU(),
                                                nn.Linear(4 * d_model, d_model))
                                  for _ in range(n_blocks)])
        self.out = nn.Linear(d_model, 10)

    def forward(self, tokens):
        n = tokens.shape[1]
        x = self.tok(tokens) + self.pos(torch.arange(n))
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool), 1)
        for attn, mlp in zip(self.attn, self.mlp):
            x = x + attn(x, x, x, attn_mask=mask, need_weights=False)[0]
            x = x + mlp(x)
        return self.out(x)

def batch(rng, size, width):
    a, b = rng.integers(0, 10, (size, width)), rng.integers(0, 10, (size, width))
    tokens = np.concatenate([np.full((size, 1), BOS), 10 * a + b,
                             np.zeros((size, 1), dtype=int)], axis=1)
    place = 10 ** np.arange(width)
    total = (a * place).sum(1).astype(object) + (b * place).sum(1).astype(object)
    digits = np.stack([np.array([(int(t) // 10**i) % 10 for t in total])
                       for i in range(width + 1)], 1)
    return torch.tensor(tokens), torch.tensor(digits)

rng = np.random.default_rng(0)
model, steps = Learned(), 6000
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)

for step in range(steps):
    width = int(rng.integers(1, 13))                  # every width from 1 to 12
    tokens, digits = batch(rng, 256, width)
    loss = nn.functional.cross_entropy(model(tokens)[:, 1:].reshape(-1, 10), digits.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 1500 == 0:
        print(f"step {step:5d}  loss {loss.item():.4f}")
print(f"final loss {loss.item():.4f}")

Now ask for one more digit than it ever saw.

In [ ]:
@torch.no_grad()
def exact_match(model, width, n=500):
    tokens, digits = batch(rng, n, width)
    return float((model(tokens)[:, 1:].argmax(-1) == digits).all(1).float().mean())

for w in (4, 8, 12, 13, 14, 16, 20):
    tag = "trained" if w <= 12 else "unseen "
    print(f"width {w:2d} [{tag}]  learned {exact_match(model, w):.3f}   hand-built 1.000")

Exact through 12, then a cliff. The obvious suspect is the positional
embedding, whose rows for unseen positions never received a gradient. That
explanation is testable and it accounts for one digit: swapping the learned table
for fixed sinusoids, a formula defined at every position, lifts width 13 and then
falls off the same cliff into the same zero.

The hand-built model has nine dimensions, no training, and no length limit beyond
the precision of the tie-break. The trained model has 64 dimensions, a perfect
in-distribution score, and stops at 13. That gap is the point of the exercise.

## Exercises

1. **Sort a sequence.** The reverser's query matrix is a permutation you wrote
   down. Sorting is a permutation you have to compute. Build a head that attends
   from each position to the position holding the next-smallest token, using the
   same "rank by predicate, tie-break by position" key pattern as the carry lookup.

2. **Make counting length-free.** The counter above bakes `seq_len` into its MLP.
   RASP's trick is to attend to the BOS token *plus* every match, so BOS receives
   weight `1/(k+1)` for `k` matches. Build that, then work out what the MLP has to
   compute to invert it, and where the construction starts to lose precision.

3. **Find the trained model's length bound.** Train at widths 1 to 12, then probe
   at 13 with inputs of your choosing. Are the failures concentrated in the top
   columns, the bottom columns, or the long carry chains? Does the model's block-1
   attention still converge the way it does at width 12?

4. **Break the adder on purpose.** Lower `beta` until the adder starts making
   mistakes, and check whether the first errors appear on long carry chains,
   as the tie-break argument predicts.

5. **Subtraction.** Borrowing is the mirror of carrying, with a *borrow-generate*
   and *borrow-propagate* pair. Write it, and reuse everything but the two
   indicators in MLP 0.